[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/01_Python%E5%9F%BA%E7%A4%8E/08_%E3%82%B9%E3%83%97%E3%83%AC%E3%83%83%E3%83%89%E3%82%B7%E3%83%BC%E3%83%88%E3%81%8B%E3%82%89Python%E3%81%B8.ipynb)

# Python基礎-08. スプレッドシートからPythonへ

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

ふだん **Excel / Googleスプレッドシート** で見ているデータを、Pythonに取り込んで分析する方法です。「いつものスプレッドシート」と「Python」を**つなぐ**のがこの章のねらいです。

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import os
# ローカルに無ければ公開リポジトリ(raw)からExcelを読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
xlsx = '../data/marketing.xlsx' if os.path.exists('../data/marketing.xlsx') else RAW + 'marketing.xlsx'
print('読み込み元:', xlsx)

## この章でできるようになること
- Excelブックのデータを Python に読み込める（`read_excel`）
- 1つのブックの**複数シート**を扱える
- Excelの**ピボット集計**を pandas の `groupby` で再現できる
- Googleスプレッドシートを URL で読み込める
- 分析結果を **Excelに書き出せる**（`to_excel`）

**前提**：`01_Python基礎/06 pandas入門`　/　**所要**：約25分

## 1. Excelブックを読む（`read_excel`）

教材の `marketing.xlsx` は、商談・Webマーケ・ABテスト・月次KPI・顧客 の5シートが入った**1つのブック**です（実務でよくある“タブで分けた1ファイル”の形）。

In [ ]:
deals = pd.read_excel(xlsx, sheet_name='商談')   # 「商談」シートを読み込む
deals.head()

### 「商談」シート（架空のBtoB商談400件）
| 列 | 意味 |
|---|---|
| 商談ID | 商談の番号 |
| 受注日 | 日付 |
| 業界 | 顧客の業界 |
| 獲得チャネル | 展示会/Web広告/メルマガ/紹介/テレアポ |
| 商談金額 | 円 |
| 担当者 | 営業担当 |
| 受注 | 1=成約 / 0=失注 |

## 2. シート一覧と全シート読み込み

In [ ]:
xls = pd.ExcelFile(xlsx)
print('シート名:', xls.sheet_names)            # ブックに入っているシート一覧

all_sheets = pd.read_excel(xlsx, sheet_name=None)   # 全シートをまとめて辞書で取得
all_sheets['顧客'].head()                       # 「顧客」シートの先頭

## 3. Excelのピボットを pandas で再現する

Excelの「ピボットテーブルで**チャネル別の受注金額合計**」は、pandasの `groupby` で1行です。

In [ ]:
won = deals[deals['受注'] == 1]                 # 受注した商談だけ
by_ch = won.groupby('獲得チャネル')['商談金額'].sum().sort_values(ascending=False)
print(by_ch)

> **Pythonの嬉しさ**：同じ集計を毎回ピボットで作り直す必要がない。データが更新されても、このセルを実行するだけで結果が更新されます（再現性）。

## 4. Googleスプレッドシートを読む（URL）

Googleスプレッドシートは、共有URLを **CSV書き出しURL** に変えると `read_csv` で読めます。

```
https://docs.google.com/spreadsheets/d/<ファイルID>/export?format=csv&gid=<シートのgid>
```

手順：対象シートを開く → URL の `/edit#gid=...` の `<ファイルID>` と `gid` を上の形に当てはめる。

In [ ]:
# 例（自分のスプレッドシートのIDとgidに置き換えて実行）:
# url = 'https://docs.google.com/spreadsheets/d/XXXXXXXX/export?format=csv&gid=0'
# my_df = pd.read_csv(url)
# my_df.head()
print('↑ コメントを外し、自分のスプレッドシートのURLに変えて使います')

> **注意**：URLで読むには、そのスプレッドシートが「リンクを知っている全員が閲覧可」または「ウェブに公開」されている必要があります。**社外秘・個人情報のデータは公開しないこと。**

## 5. 結果をExcelに書き出す（`to_excel`）

分析結果を「Excelで」上司や同僚に渡したいとき。

In [ ]:
summary = won.groupby('獲得チャネル')['商談金額'].agg(['count', 'sum'])  # 件数と合計
summary.columns = ['受注件数', '売上合計']
summary.to_excel('チャネル別集計.xlsx')          # Excelファイルに書き出し
print('チャネル別集計.xlsx を書き出しました')

## 確認テスト（自動採点）

`ans = None` を自分の計算で置き換えて実行すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: 商談の件数（deals の行数）を ans に
ans = None   # 例: len(deals)
_check('Q1 商談件数', ans, len(deals))

In [ ]:
# Q2: 受注した商談の件数を ans に
ans = None   # 例: (deals['受注'] == 1).sum()
_check('Q2 受注件数', ans, int((deals['受注'] == 1).sum()))

---
## 練習問題

**問1.** `Webマーケ` シートを読み、チャネル別の **クリック数の合計** を出そう。

**問2.** `顧客` シートを読み、**業界別の累計売上の平均** を出そう。

**問3.** 問2の結果を `to_excel` で `業界別売上.xlsx` に書き出そう。

In [ ]:
# 問1〜3


> **解答例は別ページ**（ネタバレ防止）**[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/01_Python%E5%9F%BA%E7%A4%8E/08_%E3%82%B9%E3%83%97%E3%83%AC%E3%83%83%E3%83%89%E3%82%B7%E3%83%BC%E3%83%88%E3%81%8B%E3%82%89Python%E3%81%B8.md)**

## 用語集 ＆ チートシート

| やりたいこと | コード |
|---|---|
| 1シートを読む | `pd.read_excel('book.xlsx', sheet_name='商談')` |
| シート名一覧 | `pd.ExcelFile('book.xlsx').sheet_names` |
| 全シート読む | `pd.read_excel('book.xlsx', sheet_name=None)` |
| ピボット集計 | `df.groupby('列')['値'].sum()` |
| Excelに書き出す | `df.to_excel('out.xlsx', index=False)` |
| Googleスプレッドシート | `.../export?format=csv&gid=...` を `read_csv` |

In [ ]:
# チートシート（スプレッドシート I/O）
pd.read_excel(xlsx, sheet_name='Webマーケ')        # シート指定で読む
pd.ExcelFile(xlsx).sheet_names                     # シート一覧
deals.groupby('業界')['商談金額'].sum()             # ピボット相当
deals.head().to_excel('sample.xlsx', index=False)  # 書き出し

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "01_Python基礎/08_スプレッドシートからPythonへ"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")